###Run Shared Libraries

In [0]:
%run ../Misc/sharedlibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimparty"

###Read Bronze table

In [0]:
partiesDf = spark.table("bronze.parties")
partyaddressDf = spark.table("bronze.partyaddress")

###Verify Bronze Schemas

In [0]:
partiesDf.printSchema()
partyaddressDf.printSchema()

###Create Latest Party Address DataFrame

In [0]:
partyAddressWindow = Window.partitionBy("PartyNumber").orderBy(
    F.col("ValidFrom").desc(),
    F.col("RecordId").desc()
)

partyaddressLatestDf = (
    partyaddressDf
    .withColumn("rn", F.row_number().over(partyAddressWindow))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

###Validate Latest Party Address Logic

###Build Dimension/Fact table

In [0]:
dimPartyDf = partiesDf.join(
    partyaddressLatestDf, partiesDf.PartyId == partyaddressLatestDf.PartyNumber, "left"
    ).filter(partiesDf.RecordId.isNotNull()
    ).select(
        partiesDf.PartyId,
        F.trim(partiesDf.PartyName).alias("PartyName"),
        F.when(partiesDf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(partiesDf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
        F.from_utc_timestamp(partiesDf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
        partiesDf.PartyAddressCode.alias("PartyAddressCode"),
        F.from_utc_timestamp(partiesDf.EstablishedDate,'CST').alias("EstablishedDate"),
        F.trim(partiesDf.PartyEmailId).alias("PartyEmailId"),
        F.trim(partiesDf.PartyContactNumber).alias("PartyContactNumber"),
        partiesDf.RecordId.alias("PartyRecordId"),
        F.trim(partiesDf.TaxId).alias("TaxId"),
        F.trim(partyaddressLatestDf.Address).alias("Address"),
        F.trim(partyaddressLatestDf.City).alias("City"),
        F.trim(partyaddressLatestDf.State).alias("State"),
        F.trim(partyaddressLatestDf.Country).alias("Country"),
        F.trim(partyaddressLatestDf.ZipCode).alias("ZipCode"),
        F.trim(partyaddressLatestDf.Region).alias("Region"),
        F.from_utc_timestamp(partyaddressLatestDf.ValidFrom,'CST').alias("ValidFrom"),
        F.when(partyaddressLatestDf.ValidTo.isNull(), F.lit("1900-01-01")).otherwise(partyaddressLatestDf.ValidTo).cast("timestamp").alias("ValidTo"),
        partyaddressLatestDf.RecordId.alias("PartyAddressRecordId")
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("PartyHashKey", F.xxhash64("PartyRecordId")
    )

display(dimPartyDf)

###Validate Dimension Before Write

In [0]:
print("dimPartyDf count:", dimPartyDf.count())

display(
    dimPartyDf
    .groupBy("PartyId")
    .count()
    .filter(F.col("count") > 1)
)

display(
    dimPartyDf
    .filter(F.col("PartyId") == 10022)
    .orderBy("PartyRecordId", "PartyAddressRecordId")
)

###Final dataframe

In [0]:
df_final = dimPartyDf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final,"silver",Entity)

###Validate Silver Table After Write

In [0]:
display(spark.table("silver.dimparty"))

In [0]:
display(
    spark.table("silver.dimparty")
    .groupBy("PartyId")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
display(
    spark.table("silver.dimparty")
    .filter(F.col("PartyId") == 10022)
    .orderBy("PartyRecordId", "PartyAddressRecordId")
)

In [0]:
dimPartyDf.printSchema()

In [0]:
partyaddressDf.printSchema()